In [1]:
from dotenv import load_dotenv
import os
load_dotenv()
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
QDRANT_URL = os.getenv("QDRANT_URL")
DATABASE_NAME = os.getenv("DATABASE_NAME")

In [22]:
from langchain.chat_models import init_chat_model

def get_model():
    return init_chat_model(model="groq:llama-3.3-70b-versatile")

model = get_model()

In [60]:
from langchain.chat_models import init_chat_model

def get_streaming_model():
    return init_chat_model(model="groq:llama-3.3-70b-versatile",streaming=True)

streaming_model = get_streaming_model()

In [2]:
# create embedding model

from langchain_huggingface import HuggingFaceEmbeddings


def get_embedding_model():
    return HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

embedding_model = get_embedding_model()

/Users/amanchandra/Desktop/rag/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12295.46it/s]


In [3]:
# get vector size
def get_vector_size():
    return len(
        embedding_model.embed_query("Hello")
    )

vector_size = get_vector_size()

In [4]:
#create qdrant client

from qdrant_client import QdrantClient
import qdrant_client
def get_qdrant_client():
    return QdrantClient(
        url=QDRANT_URL,
        api_key=QDRANT_API_KEY
    )
    
client = get_qdrant_client()
client.get_collections()
# print(qdrant_client.__version__)

CollectionsResponse(collections=[CollectionDescription(name='enterprise_docs'), CollectionDescription(name='knowledge_base')])

In [5]:
#create collection if it doesn't exixts

from qdrant_client.models import VectorParams,Distance

def create_collection_if_not_exists():
    collections = client.get_collections().collections

    collections_name = [collection.name for collection in collections]

    if DATABASE_NAME not in collections_name:
        client.create_collection(
            collection_name=DATABASE_NAME,
            vectors_config=VectorParams(
                size=vector_size,
                distance=Distance.COSINE
            )
        )

        print("✅ Collection Created")

       

        print("✅ Payload Indexes Created")
    else:
        print("ℹ️ Collection Already Exists")

create_collection_if_not_exists()


✅ Collection Created
✅ Payload Indexes Created


In [6]:
 
from qdrant_client.models import PayloadSchemaType
 # Create payload index for department
client.create_payload_index(
    collection_name=DATABASE_NAME,
    field_name="department",
    field_schema=PayloadSchemaType.KEYWORD
)

# Create payload index for source
client.create_payload_index(
    collection_name=DATABASE_NAME,
    field_name="source",
    field_schema=PayloadSchemaType.KEYWORD
)

UpdateResult(operation_id=4, status=<UpdateStatus.COMPLETED: 'completed'>)

In [136]:
# create vector store

from langchain_qdrant import QdrantVectorStore


def create_vector_store():
    return QdrantVectorStore(
        client=client,
        embedding=embedding_model,
        collection_name=DATABASE_NAME
    )

vector_store = create_vector_store()

In [7]:
# create ingestion pipeline

import os
import uuid
from qdrant_client.models import PointStruct
from langchain_community.document_loaders import PyPDFLoader, pdf
from langchain_text_splitters import RecursiveCharacterTextSplitter

def ingest_document(pdf_path,department):
    # load pdf
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()

    # split into chunks
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=100
    )
    chunks = text_splitter.split_documents(documents)

    # create points
    points=[]
    for chunk in chunks:
        vector = embedding_model.embed_query(chunk.page_content)
        point = PointStruct(
            id=str(uuid.uuid4()),
            vector=vector,
            payload={
                "page_content":chunk.page_content,
                "department":department,
                "source":os.path.basename(pdf_path),
                "page":chunk.metadata.get("page")
            }
        )

        points.append(point)

    # upload to qdrant
    client.upsert(
        collection_name=DATABASE_NAME,
        wait=True,
        points=points
    )
    

    print("=" * 50)
    print("📄 File:", os.path.basename(pdf_path))
    print("📑 Pages:", len(documents))
    print("Departmen:", department)
    print("✂️ Chunks:", len(chunks))
    print("✅ Uploaded Successfully")
    print("=" * 50)


/var/folders/_z/kwjgrstn4rb6ht4c1764f8gw0000gn/T/ipykernel_44158/1862844291.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, pdf


In [8]:
ingest_document("data/Grammer.pdf",department="Language")
ingest_document("data/Insurnace.pdf",department="Insurance")
ingest_document("data/sample-return-refund-policy-template.pdf",department="Refund")

Ignoring wrong pointing object 5 0 (offset 0)


📄 File: Grammer.pdf
📑 Pages: 6
Departmen: Language
✂️ Chunks: 53
✅ Uploaded Successfully
📄 File: Insurnace.pdf
📑 Pages: 5
Departmen: Insurance
✂️ Chunks: 23
✅ Uploaded Successfully
📄 File: sample-return-refund-policy-template.pdf
📑 Pages: 3
Departmen: Refund
✂️ Chunks: 11
✅ Uploaded Successfully


In [15]:
from qdrant_client.models import Filter, FieldCondition, MatchValue

def retrieve_documents(query_vector, department=None, k=3):

    query_filter = None

    if department:
        query_filter = Filter(
            must=[
                FieldCondition(
                    key="department",
                    match=MatchValue(value=department)
                )
            ]
        )

    retrieved_docs = client.query_points(
        collection_name=DATABASE_NAME,
        query=query_vector,
        limit=k,
        query_filter=query_filter,
        with_payload=True
    )

    return retrieved_docs.points

In [16]:
def embed_query(query:str):
    """
    Convert user query into embedding vector
    """

    query_vector = embedding_model.embed_query(query)

    return query_vector

In [50]:
query = "Tell me the details of Cashless Medical Coverage"
# query = "Tell me the dependency based rules"

query_vector = embed_query(query)

results = retrieve_documents(query_vector,department="Insurance")

print(f"Results: {results}")

for doc in results:
    print(doc.payload["page_content"])
    print(doc.payload["department"])
    print(doc.payload["source"])
    # print(doc.payload["metadata"])
    print("=" *80)


Results: [ScoredPoint(id='dfb0d2f8-7ec0-45cf-ac67-a6d6d9b56dfd', version=6, score=0.60459274, payload={'page_content': 'for Final Approval.  After scrutinizing the \ndocument Cashless team will approved the \nFinal Amount as per Bill given.  \nNow Patient can pay only the Non \nadmissible Item as mention in the Final Bill. \nAfter receiving the physical document from Hospital \nMedi Assist will Pay the Claim Amount Directly to \nHospital Authority.  \nObtain  Referral/Authorization slip- \nissued by health center', 'department': 'Insurance', 'source': 'Insurnace.pdf', 'page': 2}, vector=None, shard_key=None, order_value=None), ScoredPoint(id='f0ad5427-1acc-47a1-aaaa-fcc35ad70ee1', version=6, score=0.5938382, payload={'page_content': 'by/following hospitalization). Cashless \nhospitalization is “facility in empanelled \nhospitals”. All the leading hospitals in & \naround Dhanbad (Asarfi Hospital, Jalan \nHospital, PMCH, JIMS etc.), Bokaro, \nAsansol, Durgapur (Mission Hospital, City \nH

In [18]:
# create context

def build_context(points):
    context=""
    for index,point in enumerate(points,start=1):
        context += f"""
        Document {index}
        
        {point.payload["page_content"]}

        ---------------------------------
        
        """
    
    return context

In [20]:
context = build_context(results)
print(context)


        Document 1

        for Final Approval.  After scrutinizing the 
document Cashless team will approved the 
Final Amount as per Bill given.  
Now Patient can pay only the Non 
admissible Item as mention in the Final Bill. 
After receiving the physical document from Hospital 
Medi Assist will Pay the Claim Amount Directly to 
Hospital Authority.  
Obtain  Referral/Authorization slip- 
issued by health center

        ---------------------------------

        
        Document 2

        by/following hospitalization). Cashless 
hospitalization is “facility in empanelled 
hospitals”. All the leading hospitals in & 
around Dhanbad (Asarfi Hospital, Jalan 
Hospital, PMCH, JIMS etc.), Bokaro, 
Asansol, Durgapur (Mission Hospital, City 
Hospital, Disha Eye Hospital etc.) All types 
of diseases, including Eye, Ear, and pre-
existing diseases, shall also be covered. 
(Dental care  will only be entertained in 
accidental case) 
Rs.2,50,000/- per Annum.

        -------------------------

In [21]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are an Enterprise Knowledge Assistant.

Answer ONLY using the provided context.

If the answer cannot be found in the context,
reply with:

"I don't have enough information."

Context:
{context}

Question:
{question}

Answer:
""")

In [ ]:
def generate_answer(question,context):
    chain = prompt | model

    response = chain.invoke({
        "context":context,
        "question":question
    })
    # print(response)
    return response.content
    


In [ ]:
def generate_streaming_answer(question,context):
    chain = prompt | streaming_model

    final_answer=""

    for chunk in chain.stream({

        "question": question,

        "context": context

    }):

        # print(chunk.content, end="|", flush=True)

        final_answer += chunk.content

    print()

    
    # print(response)
    return final_answer

In [71]:
answer = generate_answer("Tell me the details of Cashless Medical Coverage",context)

print(answer)

Cashless Medical Coverage includes medical expenses per annum for student (indoor/inpatient treatment, any type of accidental treatment/test required, followed by/following hospitalization). Cashless hospitalization is a "facility in empanelled hospitals". The coverage amount is Rs.2,50,000/- per Annum and it covers all types of diseases, including Eye, Ear, and pre-existing diseases. (Dental care will only be entertained in accidental case)


In [66]:
chat_history=[]

def save_messages(role,content):
    chat_history.append(
        {
            "role":role,
            "content":content
        }
    )

In [37]:
def get_chat_history():

    history=""

    for message in chat_history:
        history += f"{message["role"].capitalize()}: {message["content"]}\n"

    return history

In [43]:
from langchain_core.prompts import ChatPromptTemplate

query_rewrite_prompt = ChatPromptTemplate.from_template("""
You are an expert search query optimizer.

Your job is to rewrite the user's question into a clear,
complete, standalone search query.

Rules:

- Preserve the original meaning.
- Do not answer the question.
- Add missing context when obvious.
- Return ONLY the rewritten query.

Question:
{question}

Rewritten Query:
""")

In [38]:
from langchain_core.prompts import ChatPromptTemplate

rewrite_prompt = ChatPromptTemplate.from_template("""
You are an AI assistant.

Given the conversation history and the latest user question,
rewrite the latest question into a standalone question.

Do NOT answer it.

Conversation History:

{history}

Current Question:

{question}

Standalone Question:
""")

In [39]:
def rewrite_question(question):
    chain = rewrite_prompt | model

    response = chain.invoke({
        "history":get_chat_history(),
        "question":question
    })

    return response.content

In [44]:
def rewrite_query(question):

    chain = query_rewrite_prompt | model

    response = chain.invoke({
        "question": question
    })

    return response.content.strip()

In [53]:
def build_citations(points):
    citations = []

    for point in points:
        payload = point.payload
        citation = {
            "source":payload["source"],
            "page":payload["page"]+1,
            "department":payload["department"],
            "score":round(point.score,3)
        }
        citations.append(citation)

    return citations

In [51]:
def remove_duplicate_citations(citations):

    unique = []

    seen = set()

    for citation in citations:

        key = (
            citation["source"],
            citation["page"]
        )

        if key not in seen:

            seen.add(key)

            unique.append(citation)

    return unique

In [72]:
def ask(question,department=None):

    rewritten = rewrite_query(question)

    standalone_question = rewrite_question(rewritten)

    # step 1
    query_vector = embed_query(standalone_question)

    # step 2
    retrieved_docs = retrieve_documents(
                        query_vector,
                        department
                    )

    # step 3
    context = build_context(retrieved_docs)

    # step 4
    # answer = generate_answer(question,context)
    answer = generate_streaming_answer(question,context)

    save_messages("user",question)
    save_messages("assistant",answer)

    citations = build_citations(retrieved_docs)

    citations = remove_duplicate_citations(citations)

    return {
        "question":question,
        "answer":answer,
        "citations":citations
    }

In [76]:
response = ask(
    question="Tell me about Cashless Medical Coverage",
    department="Insurance"
)

print(response["answer"])

|Cash|less| Medical| Coverage|:| Medical| expenses| per| annum| for| student| (|ind|oor|/in|patient| treatment|,| any| type| of| accidental| treatment|/test| required|,| followed| by|/f|ollow|ing| hospital|ization|).| Cash|less| hospital|ization| is| “|facility| in| em|panel|led| hospitals|”.| All| types| of| diseases|,| including| Eye|,| Ear|,| and| pre|-existing| diseases|,| shall| also| be| covered|.| (|D|ental| care| will| only| be| entertained| in| accidental| case|)| The| coverage| amount| is| Rs|.|2|,|50|,|000|/-| per| An|num| and| Rs|.| |1|,|27|,|000|/-| per| semester|,| approx|.,| or| may| vary| as| per| the| institute| policy|.|||
Cashless Medical Coverage: Medical expenses per annum for student (indoor/inpatient treatment, any type of accidental treatment/test required, followed by/following hospitalization). Cashless hospitalization is “facility in empanelled hospitals”. All types of diseases, including Eye, Ear, and pre-existing diseases, shall also be covered. (Dental car

In [59]:
response = ask(
    "Who can avail it?",
    department="Insurance"
)

# print(response["rewritten_question"])

print(response["citations"])

[{'source': 'Insurnace.pdf', 'page': 3, 'department': 'Insurance', 'score': 0.563}, {'source': 'Insurnace.pdf', 'page': 1, 'department': 'Insurance', 'score': 0.526}]


In [28]:
print("\nSources\n")

for point in response["sources"]:

    print("=" * 80)

    print("Department :", point.payload["department"])

    print("Source :", point.payload["source"])

    print("Page :", point.payload["page"])

    print()

    print(point.payload["page_content"][:300])

    print()


Sources

Department : Insurance
Source : Insurnace.pdf
Page : 2

for Final Approval.  After scrutinizing the 
document Cashless team will approved the 
Final Amount as per Bill given.  
Now Patient can pay only the Non 
admissible Item as mention in the Final Bill. 
After receiving the physical document from Hospital 
Medi Assist will Pay the Claim Amount Directl

Department : Insurance
Source : Insurnace.pdf
Page : 0

by/following hospitalization). Cashless 
hospitalization is “facility in empanelled 
hospitals”. All the leading hospitals in & 
around Dhanbad (Asarfi Hospital, Jalan 
Hospital, PMCH, JIMS etc.), Bokaro, 
Asansol, Durgapur (Mission Hospital, City 
Hospital, Disha Eye Hospital etc.) All types 
of di

Department : Insurance
Source : Insurnace.pdf
Page : 0

actual basis, i.e., Rs. 1,27,000/- per 
semester, approx., or may vary as per 
the institute policy 
 
2 Claim in case of accidental death or 
incapacitation/permanent disability 
of the student. 
Rs. 5,00,000/- (Five)